# Calibración del simulador

Convierte los datos de la hoja de observación de campo en los tres JSONs que lee el simulador.

**Completar en orden después de la visita de campo.**

## Setup

In [ ]:
import json
from pathlib import Path

FLUJOS_PATH = Path('data/processed/flujos_calibrados.json')
MARKOV_PATH  = Path('data/processed/matriz_markov.json')
GEO_PATH     = Path('data/processed/geometria_carriles.json')

flujos = json.loads(FLUJOS_PATH.read_text())
markov = json.loads(MARKOV_PATH.read_text())
geo    = json.loads(GEO_PATH.read_text())

print('JSONs cargados correctamente')

## A. Tiempos del semáforo

Llenar con los promedios medidos de la **sección A** de la hoja.
Medir mínimo 3 ciclos y promediar.

In [ ]:
# ── COMPLETAR CON DATOS DE CAMPO (sección A) ──
VERDE_TOLUCA     = None   # ej. 42
VERDE_PERIFERICO = None   # ej. 58
AMARILLO         = None   # ej. 4

flujos['tiempos_semaforo_observados']['verde_toluca_s']     = VERDE_TOLUCA
flujos['tiempos_semaforo_observados']['verde_periferico_s'] = VERDE_PERIFERICO
flujos['tiempos_semaforo_observados']['amarillo_s']         = AMARILLO
flujos['tiempos_semaforo_observados']['ciclo_total_s']      = (
    (VERDE_TOLUCA or 0) + (VERDE_PERIFERICO or 0) + (AMARILLO or 0) * 2
)

FLUJOS_PATH.write_text(json.dumps(flujos, indent=2, ensure_ascii=False))
print('Tiempos guardados:', flujos['tiempos_semaforo_observados'])

## B. Flujos vehiculares

Desde los conteos de la **sección C.1**.

Fórmula: `veh/hora = (total_en_5min / 5) * 60`

In [ ]:
# ── COMPLETAR CON CONTEOS DE CAMPO (sección C.1) ──
# Total de vehículos contados en ventana de 5 minutos
CONTEOS_5MIN = {
    'toluca_arriba':    None,
    'toluca_abajo':     None,
    'periferico_norte': None,
    'periferico_sur':   None,
}

# Periodo de la visita: 0=madrugada 1=manana_pico 2=mediodia 3=tarde_pico 4=noche
PERIODO = 1

veh_hora = {v: round((c / 5) * 60) for v, c in CONTEOS_5MIN.items() if c}
print('Flujos estimados (veh/hora):', veh_hora)

for v, fh in veh_hora.items():
    flujos['flujos_veh_hora'][v][PERIODO] = fh

FLUJOS_PATH.write_text(json.dumps(flujos, indent=2, ensure_ascii=False))
print('Flujos guardados.')

## C. Tasa de descarga

Sección **E** de la hoja: promedio de las 3 mediciones dividido entre 60.

In [ ]:
# ── COMPLETAR CON MEDICIONES DE CAMPO (sección E) ──
# (medicion_1 + medicion_2 + medicion_3) / 3 / 60
TASAS = {
    'toluca_arriba':    None,   # ej. 0.48
    'toluca_abajo':     None,
    'periferico_norte': None,
    'periferico_sur':   None,
}

flujos['tasa_descarga_veh_por_segundo'].update(TASAS)
FLUJOS_PATH.write_text(json.dumps(flujos, indent=2, ensure_ascii=False))
print('Tasas guardadas:', flujos['tasa_descarga_veh_por_segundo'])

## D. Perfiles de conductor

Estimación visual de la **sección D**. Los tres valores deben sumar 1.0.

In [ ]:
# ── COMPLETAR CON ESTIMACIÓN DE CAMPO (sección D) ──
AGRESIVO  = None   # ej. 0.22
NORMAL    = None   # ej. 0.63
CAUTELOSO = None   # ej. 0.15

assert AGRESIVO and NORMAL and CAUTELOSO, 'Llenar los tres perfiles'
assert abs(AGRESIVO + NORMAL + CAUTELOSO - 1.0) < 1e-6, 'Deben sumar 1.0'

flujos['perfiles_conductor']['agresivo']  = AGRESIVO
flujos['perfiles_conductor']['normal']    = NORMAL
flujos['perfiles_conductor']['cauteloso'] = CAUTELOSO

FLUJOS_PATH.write_text(json.dumps(flujos, indent=2, ensure_ascii=False))
print('Perfiles guardados.')

## E. Matriz de Markov

Distribución de giros de la **sección C.2**.

Fórmula: `p_movimiento = conteo_movimiento / total_vialidad`

In [ ]:
# ── COMPLETAR CON CONTEOS DE CAMPO (sección C.2) ──
# Para cada vialidad: conteos de recto, izquierda, derecha
GIROS = {
    'toluca_arriba':    {'recto': None, 'izquierda': None, 'derecha': None},
    'toluca_abajo':     {'recto': None, 'izquierda': None, 'derecha': None},
    'periferico_norte': {'recto': None, 'izquierda': None, 'derecha': None},
    'periferico_sur':   {'recto': None, 'izquierda': None, 'derecha': None},
}

for via, g in GIROS.items():
    if not all(g.values()):
        continue
    total = sum(g.values())
    p = [round(g[m] / total, 4) for m in ['recto', 'izquierda', 'derecha']]
    p[-1] = round(1.0 - p[0] - p[1], 4)   # forzar suma = 1.0
    markov[via][str(PERIODO)] = p
    print(f'{via}: recto={p[0]:.2f}  izq={p[1]:.2f}  der={p[2]:.2f}')

MARKOV_PATH.write_text(json.dumps(markov, indent=2, ensure_ascii=False))
print('Matriz de Markov guardada.')

## F. Geometría de carriles

Datos de la **sección B** + confirmación con OSM.

In [ ]:
# ── COMPLETAR CON MEDICIONES DE CAMPO (sección B) ──
import datetime

# Longitud promedio de vehículo observada en campo
geo['longitud_vehiculo_promedio_m'] = None   # ej. 4.5

# Editar cada vialidad con sus carriles reales
# Ejemplo para toluca_arriba:
# geo['vialidades']['toluca_arriba']['carriles'][0]['longitud_almacenamiento_m'] = 75.0
# geo['vialidades']['toluca_arriba']['carriles'][0]['ancho_estimado_m'] = 3.5

geo['fecha_calibracion'] = str(datetime.date.today())
GEO_PATH.write_text(json.dumps(geo, indent=2, ensure_ascii=False))
print('Geometria guardada.')

## G. Verificación final

In [ ]:
try:
    from sim.intersection import SimuladorCruce
    sim = SimuladorCruce.desde_calibracion()
    sim.reset()
    estado = sim.get_estado()
    print('Simulador calibrado cargado correctamente')
    print(f'  Vector de estado: {estado}')
except NotImplementedError as e:
    print(f'Aun hay datos pendientes: {e}')
except Exception as e:
    print(f'Error: {e}')